- *Can we identify parallel roles purely from network structure, without narrative context?* SØREN
    * Notes:
        - Node2vec algorithm for finding nodes which are similar in one universe.
        - Struc2vec algorithm for finding nodes across networks which occupy similar structural positions.
        - Maybe: Compute a vector of structural properties (In-degree, Out-degree, Closeness, Eigenvector / pageRank, Clustering coefficient, triadic position, etc.) for each characters, and use clustering algorithms (k-means, hierarchical clustering, Gaussian mixture models)

In [3]:
# Create the edgelist for the stuc2vec algorithm

import numpy as np
import networkx as nx 
import pickle
from util import *

with open("hp_characters.pkl", "rb") as f:   # 'rb' = read binary
    HP_data = pickle.load(f)
    
HP_network = create_network(HP_data, field_origin = 'house', field_species = 'species')

with open("lotr_characters.pkl", "rb") as f:   # 'rb' = read binary
    LOTR_data = pickle.load(f)

LOTR_network = create_network(LOTR_data, field_origin = 'culture', field_species = 'race')

G = nx.union(HP_network, LOTR_network)
mapping = {old_label: i + 1 for i, old_label in enumerate(G.nodes())}
G = nx.relabel_nodes(G, mapping)
nx.write_edgelist(G, "C:\\Users\\soere\\OneDrive\\Skrivebord\\struc2vec-master\\struc2vec-master\\graph\\graph.edgelist", data=False)

with open("mapping.pkl", "wb") as f:
    pickle.dump(mapping, f)

In [4]:
# Load the generated embedding
from gensim.models import KeyedVectors
embeddings = KeyedVectors.load_word2vec_format(f"embedding.emb", binary=False)

In [6]:
import numpy as np
from sklearn.mixture import GaussianMixture

# -----------------------------------------------------------
# Step 1: Function to select best number of GMM clusters via BIC
# -----------------------------------------------------------

def select_best_gmm_k(X, k_min=2, k_max=15):
    bic_scores = []
    gmms = []
    
    for k in range(k_min, k_max + 1):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type='full',
            random_state=42
        )
        gmm.fit(X)
        bic = gmm.bic(X)
        bic_scores.append(bic)
        gmms.append(gmm)

    best_idx = np.argmin(bic_scores)
    best_k = k_min + best_idx
    best_gmm = gmms[best_idx]
    
    print(f"Best number of clusters (BIC): {best_k}")
    return best_k, best_gmm, bic_scores

In [7]:
def construct_X(G, embedding):
    in_deg = np.array([deg for _, deg in G.in_degree()])
    out_deg = np.array([deg for _, deg in G.out_degree()])
    closesness = np.array(list(nx.closeness_centrality(G).values()))
    pagerank = np.array(list(nx.pagerank(G).values()))
    clustering = np.array(list(nx.clustering(G).values()))
    X = np.c_[np.array([in_deg, out_deg, closesness, pagerank, clustering]).T, embedding]
    return X

In [8]:
X = construct_X(G, embeddings[np.array(list(mapping.values()))-1])
Z = (X - np.mean(X, axis = 0)) / np.std(X, axis = 0)

In [9]:
node_o = mapping['Harry_Potter'] 
array = []
for node, label in mapping.items():
    try:
        array.append((node, label, np.linalg.norm(embeddings[label-1] - embeddings[node_o-1])))
    except:
        continue

sorted(array, key = lambda x : x[2])

[('Harry_Potter', 4, 0.0),
 ('Hermione_Granger', 5, 0.13534394),
 ('Abernathy', 6, 0.17400253),
 ('Dirgah_Hagrid', 202, 1.2709854),
 ('Kemen', 722, 1.4224002),
 ('Gildor_Inglorion', 746, 1.4862995),
 ('Fellowship_of_the_Ring_(group)', 751, 1.5114394),
 ('Aravorn', 765, 1.5260314),
 ('Auredhir', 840, 1.7168845),
 ('Treebeard', 971, 1.7561532),
 ('Narvi', 1008, 1.8320392),
 ('Cirion', 964, 1.8510982),
 ('Ontamo', 1022, 1.8817341),
 ('Diarmid', 1046, 1.8976988),
 ('Drúv', 1058, 1.9198279),
 ('Arthad', 817, 1.9217247),
 ('Tindómiel', 828, 1.9492766),
 ('Farmer_Hogg', 1130, 1.9742941),
 ('Dirhaborn', 1053, 1.9874642),
 ('Atanatar_II', 831, 2.0275247),
 ('Almarian', 587, 2.027592),
 ('Egnor_bo-Rimion', 1082, 2.0408375),
 ('Náli', 1150, 2.0470781),
 ('Valandur', 1096, 2.0534828),
 ("Aulendil_(Vardamir's_son)", 835, 2.0804634),
 ('Eldacar_(King_of_Arnor)', 1091, 2.085764),
 ('Ecthelion_I', 1077, 2.1257527),
 ('Hador_of_Gondor', 1223, 2.150119),
 ('Mungo_MacDuff', 328, 2.15537),
 ("Helm's_daugh

In [ ]:
print(np.linalg.norm(embeddings[mapping['Albus_Dumbledore']] - embeddings[mapping['Gandalf']]))
print(np.linalg.norm(embeddings[mapping['Harry_Potter']] - embeddings[mapping['Frodo_Baggins']]))
print(np.linalg.norm(embeddings[mapping['Tom_Riddle']] - embeddings[mapping['Sauron']]))